In [1]:
!pip uninstall -y pillow torchvision

Found existing installation: pillow 11.3.0
Uninstalling pillow-11.3.0:
  Successfully uninstalled pillow-11.3.0
Found existing installation: torchvision 0.25.0+cpu
Uninstalling torchvision-0.25.0+cpu:
  Successfully uninstalled torchvision-0.25.0+cpu


In [2]:
!pip install pillow==10.4.0 torchvision --extra-index-url https://download.pytorch.org/whl/cu121

Looking in indexes: https://pypi.org/simple, https://download.pytorch.org/whl/cu121
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 44.4 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.6/7.6 MB 89.2 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 532.3/532.3 MB 3.2 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 366.2/366.2 MB 4.7 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.1/170.1 MB 10.0 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 206.0/206.0 MB 8.3 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.4/60.4 MB 30.2 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 201.5/201.5 MB 8.6 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 95.3 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.2/90.2 MB 18.3 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━

In [3]:
from huggingface_hub import login

login("hf_token")

In [4]:
from transformers import Qwen2VLForConditionalGeneration, AutoProcessor
import torch
from PIL import Image
import os
import cv2
import pandas as pd
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.figure import Figure
from collections import defaultdict
from PIL import Image
import zipfile
import shutil
import time

In [5]:
# Загрузка модели
model_name = "Qwen/Qwen2-VL-2B-Instruct"

print("Загружаем модель... Это может занять 1-2 минуты")

model = Qwen2VLForConditionalGeneration.from_pretrained(
    model_name,
    torch_dtype=torch.bfloat16,      # или torch.float16, если мало памяти
    device_map="auto",               # автоматически использует GPU
    trust_remote_code=True,
    ignore_mismatched_sizes=True
)

processor = AutoProcessor.from_pretrained(model_name, trust_remote_code=True)

print("Модель успешно загружена!")

Загружаем модель... Это может занять 1-2 минуты


config.json: 0.00B [00:00, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/729 [00:00<?, ?it/s]

...

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Модель успешно загружена!


In [6]:
def extract_price_tag(image_path: str):
    image = Image.open(image_path).convert("RGB")
    
    # Оптимальный размер для 2B модели
    max_size = 1024
    if max(image.size) > max_size:
        ratio = max_size / max(image.size)
        new_size = tuple(int(dim * ratio) for dim in image.size)
        image = image.resize(new_size, Image.LANCZOS)
    
    prompt = """Ты — эксперт по российским ценникам из магазинов (Лента, Магнит и т.д.).
Извлеки всю информацию с ценника и верни **только** JSON.

Поля:
- product_name
- price_default
- price_card
- price_discount
- discount_amount
- barcode
- id_sku
- print_datetime
- code
- additional_info
- color
- special_symbols

Если поле отсутствует — пиши null."""

    messages = [{"role": "user", "content": [
        {"type": "image", "image": image},
        {"type": "text", "text": prompt}
    ]}]

    text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

    inputs = processor(
        text=[text],
        images=[image],
        padding=True,
        return_tensors="pt",
        min_pixels=256*28*28,
        max_pixels=1024*28*28,
    )
    
    inputs = inputs.to(model.device)

    generated_ids = model.generate(
        **inputs,
        max_new_tokens=512,
        temperature=0.1,
        do_sample=False,
        num_beams=1,
        use_cache=True
    )

    generated_text = processor.decode(generated_ids[0], skip_special_tokens=True)
    
    # Извлекаем JSON
    import re, json
    try:
        json_match = re.search(r'\{[\s\S]*\}', generated_text)
        if json_match:
            result = json.loads(json_match.group(0))
            return result
        else:
            return {"error": "JSON not found", "raw": generated_text[-1000:]}
    except Exception as e:
        return {"error": str(e), "raw": generated_text[-1000:]}

In [12]:
def extract_price_tag(image_path: str):
    image = Image.open(image_path).convert("RGB")
    
    # Сильнее уменьшаем размер — ускоряет и улучшает стабильность
    max_size = 768
    if max(image.size) > max_size:
        ratio = max_size / max(image.size)
        new_size = tuple(int(dim * ratio) for dim in image.size)
        image = image.resize(new_size, Image.LANCZOS)
    
    prompt = """Ты — эксперт по ценникам. Извлеки данные **строго** в JSON формате.
Не придумывай barcode длиннее 20 символов.
Не добавляй никакой текст кроме JSON.

{
  "product_name": "...",
  "price_default": число или null,
  "price_card": число или null,
  "price_discount": число или null,
  "discount_amount": число или null,
  "barcode": строка или null,
  "id_sku": строка или null,
  "print_datetime": строка или null,
  "code": строка или null,
  "additional_info": строка или null,
  "color": строка или null,
  "special_symbols": строка или null
}"""

    messages = [{"role": "user", "content": [
        {"type": "image", "image": image},
        {"type": "text", "text": prompt}
    ]}]

    text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

    inputs = processor(
        text=[text],
        images=[image],
        padding=True,
        return_tensors="pt",
        min_pixels=256*28*28,
        max_pixels=768*28*28,
    ).to(model.device)

    generated_ids = model.generate(
        **inputs,
        max_new_tokens=400,        # уменьшили
        temperature=0.0,
        do_sample=False,
        num_beams=1,
        use_cache=True,
        eos_token_id=processor.tokenizer.eos_token_id,
        pad_token_id=processor.tokenizer.pad_token_id,
    )

    generated_text = processor.decode(generated_ids[0], skip_special_tokens=True)
    
    # Улучшенный парсинг JSON
    import re, json
    try:
        # Ищем последний JSON в тексте
        json_matches = re.findall(r'\{[\s\S]*?\}', generated_text)
        if json_matches:
            # Берём последний найденный JSON
            json_str = json_matches[-1]
            result = json.loads(json_str)
            return result
        else:
            return {"error": "JSON not found", "raw": generated_text[-800:]}
    except Exception as e:
        return {
            "error": f"Parse error: {str(e)}", 
            "raw": generated_text[-800:]
        }

In [13]:
start_time = time.time()
result = extract_price_tag('/kaggle/input/datasets/mashamalyshkina/lenta-price-tags/price_tag_cropped0.jpg')
end_time = time.time()
print(f"Время выполнения: {end_time - start_time:.2f} секунд")
print(result)

Время выполнения: 183.23 секунд
{'product_name': 'Мед частная ПАСЕКА Фитнес', 'price_default': 319.99, 'price_card': None, 'price_discount': None, 'discount_amount': None, 'barcode': '359', 'id_sku': '319', 'print_datetime': None, 'code': '319', 'additional_info': None, 'color': None, 'special_symbols': None}


In [14]:
start_time = time.time()
result = extract_price_tag('/kaggle/input/datasets/mashamalyshkina/lenta-price-tags/price_tag_cropped10.jpg')
end_time = time.time()
print(f"Время выполнения: {end_time - start_time:.2f} секунд")
print(result)

Время выполнения: 271.28 секунд
{'error': 'Parse error: Expecting value: line 3 column 20 (char 46)', 'raw': 'ount": число или null,\n  "discount_amount": число или null,\n  "barcode": строка или null,\n  "id_sku": строка или null,\n  "print_datetime": строка или null,\n  "code": строка или null,\n  "additional_info": строка или null,\n  "color": строка или null,\n  "special_symbols": строка или null\n}\nassistant\n{\n  "product_name": "Монастырская Трапеза белое сухое (Россия) 1 л",\n  "price_default": 399,\n  "price_card": 329,\n  "price_discount": 30,\n  "discount_amount": 30,\n  "barcode": "1234567890123456789012345678901234567890123456789012345678901234567890123456789012345678901234567890123456789012345678901234567890123456789012345678901234567890123456789012345678901234567890123456789012345678901234567890123456789012345678901234567890123456789012345678901234567890123456789012345678901234567890123456789012345'}


In [15]:
start_time = time.time()
result = extract_price_tag('/kaggle/input/datasets/mashamalyshkina/lenta-price-tags/price_tag_cropped.jpg')
end_time = time.time()
print(f"Время выполнения: {end_time - start_time:.2f} секунд")
print(result)

Время выполнения: 171.44 секунд
{'product_name': 'Premium с орехами', 'price_default': None, 'price_card': None, 'price_discount': None, 'discount_amount': None, 'barcode': '244', 'id_sku': None, 'print_datetime': None, 'code': '305', 'additional_info': None, 'color': None, 'special_symbols': None}


In [21]:
start_time = time.time()
result = extract_price_tag('/kaggle/input/datasets/mashamalyshkina/lenta-price-tags/price_tag_cropped1.jpg')
end_time = time.time()
print(f"Время выполнения: {end_time - start_time:.2f} секунд")
print(result)

Время выполнения: 225.55 секунд
{'product_name': 'Мед Берестов А. С. Натуральный горный (Россия)', 'price_default': None, 'price_card': None, 'price_discount': None, 'discount_amount': None, 'barcode': '5999', 'id_sku': None, 'print_datetime': None, 'code': '469', 'additional_info': None, 'color': None, 'special_symbols': None}


In [22]:
start_time = time.time()
result = extract_price_tag('/kaggle/input/datasets/mashamalyshkina/lenta-price-tags/price_tag_cropped2.jpg')
end_time = time.time()
print(f"Время выполнения: {end_time - start_time:.2f} секунд")
print(result)

Время выполнения: 200.24 секунд
{'product_name': 'Мед частная пасека с кедровым орехом стекло (Россия)', 'price_default': 799, 'price_card': 799, 'price_discount': 0, 'discount_amount': 0, 'barcode': '945', 'id_sku': '799', 'print_datetime': '2023-01-01', 'code': '799', 'additional_info': 'Мед частная пасека с кедровым орехом стекло (Россия) 225г', 'color': 'orange', 'special_symbols': '15%'}


In [23]:
start_time = time.time()
result = extract_price_tag('/kaggle/input/datasets/mashamalyshkina/lenta-price-tags/price_tag_cropped3.jpg')
end_time = time.time()
print(f"Время выполнения: {end_time - start_time:.2f} секунд")
print(result)

Время выполнения: 194.80 секунд
{'product_name': 'Мед частная пасека', 'price_default': None, 'price_card': None, 'price_discount': None, 'discount_amount': None, 'barcode': '368', 'id_sku': None, 'print_datetime': None, 'code': '234', 'additional_info': None, 'color': None, 'special_symbols': None}


In [24]:
start_time = time.time()
result = extract_price_tag('/kaggle/input/datasets/mashamalyshkina/lenta-price-tags/price_tag_cropped4.jpg')
end_time = time.time()
print(f"Время выполнения: {end_time - start_time:.2f} секунд")
print(result)

Время выполнения: 176.63 секунд
{'product_name': 'Мед Берестов', 'price_default': None, 'price_card': None, 'price_discount': None, 'discount_amount': None, 'barcode': '514', 'id_sku': None, 'print_datetime': None, 'code': 'Мед Берестов', 'additional_info': None, 'color': None, 'special_symbols': None}


In [25]:
start_time = time.time()
result = extract_price_tag('/kaggle/input/datasets/mashamalyshkina/lenta-price-tags/price_tag_cropped5.jpg')
end_time = time.time()
print(f"Время выполнения: {end_time - start_time:.2f} секунд")
print(result)

Время выполнения: 187.38 секунд
{'product_name': 'Мед Мастер Меда Разноцветные', 'price_default': None, 'price_card': None, 'price_discount': None, 'discount_amount': None, 'barcode': '2218', 'id_sku': '130', 'print_datetime': '2218', 'code': '178', 'additional_info': 'Ты — эксперт по ценникам.', 'color': 'orange', 'special_symbols': 'Мед Мастер Меда Разноцветные'}
